In [1]:
import cv2 as cv
import numpy as np
import tensorflow as tf
import winsound
import pickle
import sys
from PyQt5.QtWidgets import *

# [1] 학습된 CNN 견종 분류 모델 불러오기
cnn = tf.keras.models.load_model('cnn_for_stanford_dogs.h5')

# [2] 클래스 이름(견종 이름) 리스트 불러오기
dog_species = pickle.load(open('dog_species_names.txt', 'rb'))

# [3] PyQt5 GUI 창 클래스 정의
class DogSpeciesRecognition(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('견종 인식')
        self.setGeometry(200, 200, 700, 100)

        # [4] 버튼 3개 생성: 파일 열기, 품종 인식, 종료
        fileButton = QPushButton('강아지 사진 열기', self)
        recognitionButton = QPushButton('품종 인식', self)
        quitButton = QPushButton('나가기', self)

        # [5] 버튼 위치 설정
        fileButton.setGeometry(10, 10, 100, 30)
        recognitionButton.setGeometry(110, 10, 100, 30)
        quitButton.setGeometry(510, 10, 100, 30)

        # [6] 버튼 이벤트 연결
        fileButton.clicked.connect(self.pictureOpenFunction)
        recognitionButton.clicked.connect(self.recognitionFunction)
        quitButton.clicked.connect(self.quitFunction)

    # [7] 파일 열기 함수
    def pictureOpenFunction(self):
        # 파일 다이얼로그로 이미지 선택
        fname = QFileDialog.getOpenFileName(self, '강아지 사진 읽기', './')
        self.img = cv.imread(fname[0])  # 이미지 읽기 (OpenCV)
        if self.img is None:
            sys.exit('파일을 찾을 수 없습니다.')  # 파일 못 찾으면 종료

        cv.imshow('Dog image', self.img)  # 원본 이미지 창에 출력

    # [8] 견종 인식 함수
    def recognitionFunction(self):
        # ResNet 입력 크기 (224, 224)로 리사이즈 & 배치 차원 추가
        x = np.reshape(cv.resize(self.img, (224, 224)), (1, 224, 224, 3))

        res = cnn.predict(x)[0]  # CNN 예측 실행 → (120, ) 확률 벡터
        top5 = np.argsort(-res)[:5]  # 확률이 높은 상위 5개 클래스 인덱스

        # 클래스 이름 매핑
        top5_dog_species_names = [dog_species[i] for i in top5]

        # 이미지에 Top5 클래스명 + 확률 표시
        for i in range(5):
            prob = '(' + str(res[top5[i]]) + ')'
            # 파일명 형태: 'n02085620-Chihuahua' → 견종명만 추출
            name = str(top5_dog_species_names[i]).split('-')[1]
            cv.putText(
                self.img,
                prob + name,
                (10, 100 + i * 30),
                cv.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

        cv.imshow('Dog image', self.img)  # 결과 이미지 재출력
        winsound.Beep(1000, 500)          # 인식 후 비프음 발생

    # [9] 종료 함수
    def quitFunction(self):
        cv.destroyAllWindows()  # OpenCV 창 닫기
        self.close()            # PyQt 창 닫기

# [10] PyQt5 애플리케이션 실행
app = QApplication(sys.argv)
win = DogSpeciesRecognition()
win.show()
app.exec_()


1/1 [==============================] - 0s 20ms/step


0

| 단계              | 내용                                            |
| --------------- | --------------------------------------------- |
| **1) 파일 열기**    | 파일 다이얼로그로 강아지 이미지 선택                          |
| **2) CNN 예측**   | 학습된 `cnn_for_stanford_dogs.h5` 모델로 120종 견종 분류 |
| **3) 결과 표시**    | 확률이 높은 상위 5개 견종 이름과 예측 확률을 이미지에 출력            |
| **4) PyQt5 UI** | 버튼 클릭으로 동작 연결: 파일 열기 → 인식 → 종료                |
| **5) winsound** | 인식 완료 후 알림음                                   |